In [8]:
import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel, AutoConfig
from datasets import load_dataset
from tqdm import tqdm
from sklearn.decomposition import PCA

###############################################################################
# Arguments
###############################################################################
args = {
    "models": [
        "nlpaueb/sec-bert-base",
        "bert-base-uncased"
    ],
    "datasets": [
        {
            "name": "yelp_review_full",
            "config": None,
            "split": "train",
            "text_column": "text",
            "label_column": "label"
        },
        {
            "name": "ag_news",
            "config": None,
            "split": "train",
            "text_column": "text",
            "label_column": "label"
        },
        {
            "name": "dbpedia_14",
            "config": None,
            "split": "train",
            "text_column": "content",
            "label_column": "label"
        },
    ],
    "max_texts": 1000,
    "batch_size": 32,
    "drift_strengths": [0.0, 0.25, 0.5, 0.75, 1.0],
    "pca_components": 2,
    "output_dir": "results_multidataset",
    "contrastive_tau": 0.07,
    "contrastive_lambda": 0.5,
    "prototype_lambda": 0.5,
    "embedding_lambda": 1.0,
    "sigma": 1.0,
}

os.makedirs(args["output_dir"], exist_ok=True)

###############################################################################
# Utility Functions
###############################################################################
def batch_generator(data, labels, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size], labels[i : i + batch_size]

def introduce_gradual_drift(text_list, fraction_shuffle=0.5):
    new_texts = []
    for txt in text_list:
        words = txt.split()
        if len(words) < 2:
            new_texts.append(txt)
            continue
        k = int(len(words) * fraction_shuffle)
        if k < 1:
            new_texts.append(txt)
            continue
        indices = list(range(len(words)))
        random.shuffle(indices)
        shuffle_indices = indices[:k]
        to_shuffle = [words[i] for i in shuffle_indices]
        random.shuffle(to_shuffle)
        for i, idx in enumerate(shuffle_indices):
            words[idx] = to_shuffle[i]
        new_texts.append(" ".join(words))
    return new_texts

def cosine_sim_torch(a, b):
    a_norm = a / (a.norm(dim=1, keepdim=True) + 1e-8)
    b_norm = b / (b.norm(dim=1, keepdim=True) + 1e-8)
    return torch.matmul(a_norm, b_norm.transpose(0,1))


In [11]:
###############################################################################
# Contrastive + Prototype Loss Functions
###############################################################################
def instance_contrastive_loss(z_a, z_b, tau=0.07):
    sim_matrix = cosine_sim_torch(z_a, z_b) / tau
    batch_size = z_a.size(0)
    labels = torch.arange(batch_size).to(z_a.device)
    loss = torch.nn.CrossEntropyLoss()(sim_matrix, labels)
    return loss

def prototype_contrastive_loss(z, proto_dict, labels, tau=0.07):
    all_protos = []
    for lbl in labels:
        all_protos.append(proto_dict[lbl].unsqueeze(0))
    all_protos = torch.cat(all_protos, dim=0)
    sim_matrix = cosine_sim_torch(z, all_protos) / tau
    idx = torch.arange(z.size(0)).to(z.device)
    loss = torch.nn.CrossEntropyLoss()(sim_matrix, idx)
    return loss

def supervised_embedding_loss(z, labels, proto_dict):
    criterion = torch.nn.MSELoss()
    losses = []
    for i, lbl in enumerate(labels):
        target_proto = proto_dict[lbl].detach().clone().requires_grad_(True)
        losses.append(criterion(z[i], target_proto))
    return torch.stack(losses).mean().requires_grad_(True)

###############################################################################
# DriftDetector Class with Class-Wise Prototypes & CLEAR-like Contrastive Training
###############################################################################
class DriftDetector(torch.nn.Module):
    def __init__(self, model_name, device, args, pca_transform=None):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.device = device
        self.args = args
        self.pca_transform = pca_transform
        self.transformer = AutoModel.from_pretrained(model_name).to(device)
        self.transformer.train()
        self.optimizer = torch.optim.Adam(self.transformer.parameters(), lr=1e-5)

        self.class_prototypes = {}
        self.prototype_history = {}
        self.class_counts = {}
        self.cosine_scores = []
        self.all_embeddings = []

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]

    def _get_embeddings(self, texts):
        encodings = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(self.device)
        return self.forward(encodings["input_ids"], encodings["attention_mask"])

    def initialize_baseline(self, texts, labels):
        self.eval()
        with torch.no_grad():
            embeddings_dict = {}
            for batch_texts, batch_labels in batch_generator(texts, labels, self.args['batch_size']):
                emb = self._get_embeddings(batch_texts)
                if self.pca_transform is not None:
                    emb = torch.from_numpy(self.pca_transform.transform(emb.cpu().numpy())).to(self.device)
                for e, lbl in zip(emb, batch_labels):
                    lbl = int(lbl)
                    if lbl not in embeddings_dict:
                        embeddings_dict[lbl] = []
                    embeddings_dict[lbl].append(e.unsqueeze(0))

            for lbl, emb_list in embeddings_dict.items():
                emb_tensor = torch.cat(emb_list, dim=0)
                proto = emb_tensor.mean(dim=0)
                self.class_prototypes[lbl] = proto
                self.prototype_history[lbl] = [proto.detach().cpu().numpy()]
                self.class_counts[lbl] = emb_tensor.size(0)
                for e in emb_list:
                    self.all_embeddings.append(e.detach().cpu().numpy())

    def detect_drifts(self, texts, labels):
        for batch_texts, batch_labels in tqdm(batch_generator(texts, labels, self.args['batch_size']), leave=False):
            batch_labels = [int(lbl) for lbl in batch_labels]
            self._contrastive_train_on_batch(batch_texts, batch_labels)
            self._approximate_drift_update(batch_texts, batch_labels)
            self._compute_cosine_score()

    def _contrastive_train_on_batch(self, batch_texts, batch_labels):
        self.train()
        self.optimizer.zero_grad()
        emb = self._get_embeddings(batch_texts).clone().detach().requires_grad_(True)

        if self.pca_transform is not None:
            emb = torch.from_numpy(self.pca_transform.transform(emb.detach().cpu().numpy())).float().to(self.device)
            emb.requires_grad_(True)

        batch_labels = [int(lbl) for lbl in batch_labels]
        emb_perm = emb[torch.randperm(emb.size(0))]
        l_cont = instance_contrastive_loss(emb, emb_perm, tau=self.args["contrastive_tau"])
        l_pro  = prototype_contrastive_loss(emb, self.class_prototypes, batch_labels, tau=self.args["contrastive_tau"])
        l_emb  = supervised_embedding_loss(emb, batch_labels, self.class_prototypes)

        loss = (self.args["embedding_lambda"] * l_emb +
                self.args["prototype_lambda"] * l_pro +
                self.args["contrastive_lambda"] * l_cont)

        loss.backward()
        self.optimizer.step()

    def _approximate_drift_update(self, batch_texts, batch_labels):
        self.eval()
        with torch.no_grad():
            emb = self._get_embeddings(batch_texts)
            if self.pca_transform is not None:
                emb = torch.from_numpy(self.pca_transform.transform(emb.cpu().numpy())).to(self.device)
            batch_labels = [int(lbl) for lbl in batch_labels]
            for e, lbl in zip(emb, batch_labels):
                if lbl not in self.class_prototypes:
                    self.class_prototypes[lbl] = e.clone()
                    self.prototype_history[lbl] = [e.detach().cpu().numpy()]
                    self.class_counts[lbl] = 1
                    continue
                delta = e - self.class_prototypes[lbl]
                w = torch.exp(-torch.norm(delta)**2 / (2.0 * self.args["sigma"]**2))
                self.class_prototypes[lbl] = self.class_prototypes[lbl] + (w * delta)
                self.prototype_history[lbl].append(self.class_prototypes[lbl].detach().cpu().numpy())
                self.class_counts[lbl] += 1
                self.all_embeddings.append(e.detach().cpu().numpy())

    def _compute_cosine_score(self):
        with torch.no_grad():
            all_protos = torch.stack(list(self.class_prototypes.values()), dim=0)
            mean_proto = all_protos.mean(dim=0)
            sims = []
            for lbl, proto in self.class_prototypes.items():
                sims.append(cosine_sim_torch(proto.unsqueeze(0), mean_proto.unsqueeze(0)).item())
            self.cosine_scores.append(np.mean(sims))

In [12]:
###############################################################################
# Full Pipeline
###############################################################################
def collect_data():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("Using device:", device)
    results = {}

    for dataset_info in args["datasets"]:
        dataset_name  = dataset_info["name"]
        dataset_config = dataset_info["config"]
        dataset_split = dataset_info["split"]
        text_col = dataset_info["text_column"]
        label_col = dataset_info["label_column"]

        print(f"\n=== Loading dataset: {dataset_name} ===")
        ds = load_dataset(dataset_name, dataset_config, split=dataset_split)
        texts = list(ds[text_col])
        labels = list(ds[label_col])

        # print(f"Sample labels: {labels[:10]}")  # Debugging print

        random.shuffle(texts)
        if args["max_texts"] > 0 and len(texts) > args["max_texts"]:
            texts = texts[: args["max_texts"]]
            labels = labels[: args["max_texts"]]

        baseline_texts = texts[: len(texts) // 2]
        drift_texts = texts[len(texts) // 2:]
        baseline_labels = labels[: len(labels) // 2]
        drift_labels = labels[len(labels) // 2:]


        for model_name in args["models"]:
            # config = AutoConfig.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)

            baseline_detector_embs = []
            with torch.no_grad():
                model_temp = AutoModel.from_pretrained(model_name).to(device).eval()
                for bt, bl in batch_generator(baseline_texts, baseline_labels, args["batch_size"]):
                    enc = tokenizer(bt, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
                    out = model_temp(**enc).last_hidden_state[:, 0, :]
                    baseline_detector_embs.append(out.cpu().numpy())
                baseline_detector_embs = np.concatenate(baseline_detector_embs, axis=0)

            pca = PCA(n_components=args["pca_components"])
            if baseline_detector_embs.shape[0] > 1:
                pca.fit(baseline_detector_embs)

            key = (dataset_info["name"], model_name)
            if key not in results:
                results[key] = []

            for drift_strength in args["drift_strengths"]:
                drifted_texts = introduce_gradual_drift(drift_texts, drift_strength)
                combined_test_texts = list(baseline_texts) + list(drifted_texts)
                combined_test_labels = list(baseline_labels) + list(drift_labels)

                detector_no_pca = DriftDetector(model_name, device, args, pca_transform=None)
                detector_no_pca.tokenizer = tokenizer
                detector_no_pca.initialize_baseline(baseline_texts, baseline_labels)
                detector_no_pca.detect_drifts(combined_test_texts, combined_test_labels)
                final_sim_no_pca = detector_no_pca.cosine_scores[-1] if len(detector_no_pca.cosine_scores) else 0.0

                detector_pca = DriftDetector(model_name, device, args, pca_transform=pca)
                detector_pca.tokenizer = tokenizer
                detector_pca.initialize_baseline(baseline_texts, baseline_labels)
                detector_pca.detect_drifts(combined_test_texts, combined_test_labels)
                final_sim_pca = detector_pca.cosine_scores[-1] if len(detector_pca.cosine_scores) else 0.0

                results[key].append({
                    "drift_strength": drift_strength,
                    "pca": False,
                    "time_series": detector_no_pca.cosine_scores[:],
                    "final_similarity": final_sim_no_pca,
                    "all_embeddings": np.array(detector_no_pca.all_embeddings)
                })
                results[key].append({
                    "drift_strength": drift_strength,
                    "pca": True,
                    "time_series": detector_pca.cosine_scores[:],
                    "final_similarity": final_sim_pca,
                    "all_embeddings": np.array(detector_pca.all_embeddings)
                })
    return results
all_results = collect_data()

Using device: mps

=== Loading dataset: yelp_review_full ===


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (1500,) + inhomogeneous part.

In [ ]:
def plot_six_subplots(all_results):
    for (dataset_name, model_name), runs in all_results.items():
        no_pca = [r for r in runs if r["pca"] is False]
        pca_   = [r for r in runs if r["pca"] is True]
        no_pca = sorted(no_pca, key=lambda x: x["drift_strength"])
        pca_   = sorted(pca_,   key=lambda x: x["drift_strength"])
        x_no_pca = [r["drift_strength"] for r in no_pca]
        y_no_pca = [r["final_similarity"] for r in no_pca]
        x_pca    = [r["drift_strength"] for r in pca_]
        y_pca    = [r["final_similarity"] for r in pca_]

        def rolling_mean_std(values, window=2):
            means, stds = [], []
            for i in range(len(values)):
                wstart = max(0, i - window + 1)
                slice_ = values[wstart:i+1]
                means.append(np.mean(slice_))
                stds.append(np.std(slice_))
            return np.array(means), np.array(stds)

        y_no_pca_rm, y_no_pca_std = rolling_mean_std(y_no_pca)
        y_pca_rm,    y_pca_std    = rolling_mean_std(y_pca)
        fig, axs = plt.subplots(2, 3, figsize=(15,8))
        axs = axs.flatten()

        axs[0].plot(x_no_pca, y_no_pca, marker='o', label='No PCA')
        axs[0].plot(x_pca, y_pca, marker='s', label='PCA')
        axs[0].set_title("Cosine Similarity vs Drift Strength")
        axs[0].set_xlabel("Drift Strength")
        axs[0].set_ylabel("Cosine Similarity")
        axs[0].legend()
        axs[0].grid(True)

        axs[1].plot(x_no_pca, y_no_pca_rm, color='blue', label='No PCA - Rolling Mean')
        axs[1].fill_between(x_no_pca,
                            y_no_pca_rm - y_no_pca_std,
                            y_no_pca_rm + y_no_pca_std,
                            color='blue', alpha=0.2)
        axs[1].plot(x_pca, y_pca_rm, color='orange', label='PCA - Rolling Mean')
        axs[1].fill_between(x_pca,
                            y_pca_rm - y_pca_std,
                            y_pca_rm + y_pca_std,
                            color='orange', alpha=0.2)
        axs[1].set_title("Rolling Mean and Std Dev")
        axs[1].set_xlabel("Drift Strength")
        axs[1].set_ylabel("Cosine Similarity")
        axs[1].legend()
        axs[1].grid(True)

        all_final_sims_no_pca = y_no_pca
        all_final_sims_pca    = y_pca
        bins = np.linspace(min(0.5, min(all_final_sims_no_pca + all_final_sims_pca)),
                           max(1.0, max(all_final_sims_no_pca + all_final_sims_pca)),
                           10)
        axs[2].hist(all_final_sims_no_pca, bins=bins, alpha=0.7, label='No PCA')
        axs[2].hist(all_final_sims_pca, bins=bins, alpha=0.7, label='PCA')
        axs[2].set_title("Histogram of Final Similarities")
        axs[2].set_xlabel("Cosine Similarity")
        axs[2].set_ylabel("Frequency")
        axs[2].legend()

        baseline_run_pca = [r for r in pca_ if r["drift_strength"] == 0.0]
        drift_run_pca    = [r for r in pca_ if r["drift_strength"] == 1.0]
        if baseline_run_pca and drift_run_pca:
            baseline_embs_2d = baseline_run_pca[0]["all_embeddings"]
            drift_embs_2d    = drift_run_pca[0]["all_embeddings"]
            axs[3].scatter(baseline_embs_2d[:, 0], baseline_embs_2d[:, 1], alpha=0.6, label="Baseline")
            axs[3].scatter(drift_embs_2d[:, 0],    drift_embs_2d[:, 1],    alpha=0.6, label="Drifted")
        axs[3].set_title("Scatter Plot (PCA Space)")
        axs[3].set_xlabel("PC1")
        axs[3].set_ylabel("PC2")
        axs[3].legend()

        baseline_sim_no_pca = y_no_pca[0] if y_no_pca else 0.0
        baseline_sim_pca    = y_pca[0]    if y_pca    else 0.0
        delta_no_pca = [sim - baseline_sim_no_pca for sim in y_no_pca]
        delta_pca    = [sim - baseline_sim_pca    for sim in y_pca]
        axs[4].plot(x_no_pca, delta_no_pca, marker='o', label='No PCA')
        axs[4].plot(x_pca, delta_pca, marker='s', label='PCA')
        axs[4].axhline(0.0, color='gray', linestyle='--', alpha=0.7)
        axs[4].set_title("Delta from Baseline Similarity")
        axs[4].set_xlabel("Drift Strength")
        axs[4].set_ylabel("Delta (Cosine Similarity)")
        axs[4].legend()
        axs[4].grid(True)

        axs[5].scatter(x_no_pca, y_no_pca, color='blue', label='No PCA', alpha=0.6)
        axs[5].scatter(x_pca, y_pca, color='orange', label='PCA', alpha=0.6)
        axs[5].set_title("Final Similarity vs Drift Strength")
        axs[5].set_xlabel("Drift Strength")
        axs[5].set_ylabel("Cosine Similarity")
        axs[5].legend()
        axs[5].grid(True)

        fig.suptitle(f"{dataset_name} | {model_name}", fontsize=16)
        fig.tight_layout()
        model_name_safe = model_name.replace("/", "_")
        fname = f"{dataset_name}_{model_name_safe}_6subplots.png"
        save_path = os.path.join(args["output_dir"], fname)
        plt.savefig(save_path)
        plt.close()

plot_six_subplots(all_results)
print("All done.")